In [14]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
import pyspark 
from pyspark.sql import SparkSession
import pandas as pd

In [4]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName("pyspark") \
        .config("hadoop.security.authentication", "simple") \
        .config("hadoop.security.authorization", "false") \
        .config("hive.metastore.use.SSL", "false") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/25 07:01:44 WARN Utils: Your hostname, codespaces-7f99a4, resolves to a loopback address: 127.0.0.1; using 10.0.1.102 instead (on interface eth0)
26/04/25 07:01:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 07:01:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-04-25 06:54:17--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.146, 18.239.115.213, 18.239.115.4, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.146|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet.1’

         fhvhv_trip   6%[>                   ]  20.48M   102MB/s               

fhvhv_tripdata_2021 100%[===================>] 294.61M  87.5MB/s    in 3.3s    

2026-04-25 06:54:20 (88.7 MB/s) - ‘fhvhv_tripdata_2021-01.parquet.1’ saved [308924937/308924937]



In [6]:
!wc -l fhvhv_tripdata_2021-01.parquet

1006794 fhvhv_tripdata_2021-01.parquet


In [5]:
df = spark.read.parquet("fhvhv_tripdata_2021-01.parquet")

df.write.mode("overwrite").csv("output_folder", header=True)

In [7]:
df.write.mode("overwrite").csv("fhvhv_tripdata_2021-01.csv", header=True)

In [8]:
df_csv = spark.read.csv("fhvhv_tripdata_2021-01.csv", header=True, inferSchema=True)

In [24]:
df_head = df_csv.limit(1001).toPandas()

In [25]:
df_head.to_csv("head.csv", index=False)

In [27]:
df_pandas = pd.read_csv("head.csv")

In [29]:
df_pandas.shape

(1001, 24)

In [33]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', StringType(), True), StructField('on_scene_datetime', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_request_flag', StringType(

Some Data cleaning just getting the columns we need 

In [36]:
from pyspark.sql import types

In [39]:
schema =  types.StructType(
    [   types.StructField("hvfhs_license_num", types.StringType(), True),
        types.StructField("dispatching_base_num", types.StringType(), True),
        types.StructField("pickup_datetime", types.TimestampType(), True),
        types.StructField("dropoff_datetime", types.TimestampType(), True),
        types.StructField("PULocationID", types.IntegerType(), True),
        types.StructField("DOLocationID", types.IntegerType(), True),
        types.StructField("SR_Flag", types.StringType(), True),
    ]
)

In [40]:
df = spark.read \
.option("header", "true") \
.schema(schema) \
.csv("fhvhv_tripdata_2021-01.csv")

Saving this csv as parquet
using repartioning (partitions) to allow spark to split the data amongst its workers

In [45]:
#lazy command doesnt yet do anything, just creates a plan for how to execute the command when we ask for the result
df = df.repartition(24)

In [46]:
df.write.parquet('fhvhv/2021/01/')

26/04/25 08:08:42 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 24, schema size: 7
CSV file: file:///workspaces/batch_processing_spark/fhvhv_tripdata_2021-01.csv/part-00001-191875d8-7911-41c0-b897-94a4831c943f-c000.csv
26/04/25 08:11:43 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 24, schema size: 7
CSV file: file:///workspaces/batch_processing_spark/fhvhv_tripdata_2021-01.csv/part-00000-191875d8-7911-41c0-b897-94a4831c943f-c000.csv
